# Datetime Handling

## Overview

Temporal data requires careful handling of parsing, arithmetic, resampling, and time zones. Pandas provides the `datetime64` dtype and a rich set of `.dt` accessor methods for vectorised datetime operations.

**Core types:**

| Type | Description |
|---|---|
| `pd.Timestamp` | Single datetime point (scalar) |
| `pd.DatetimeIndex` | Sequence of timestamps (index) |
| `pd.Timedelta` | Duration between two timestamps |
| `pd.Period` | Fixed-frequency time span (month, quarter) |

---

In [1]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

# Simulate: monitoring surveys across 3 years
n = 200
dates = pd.date_range('2021-01-01', '2023-12-31', periods=n)

surveys = pd.DataFrame({
    'site_id':     [f'SITE_{rng.integers(1,21):03d}' for _ in range(n)],
    'survey_date': rng.choice(dates, n, replace=False),
    'nitrate':     rng.gamma(2, 2, n).round(2),
    'richness':    rng.integers(5, 35, n),
}).sort_values('survey_date').reset_index(drop=True)

surveys['survey_date'] = pd.to_datetime(surveys['survey_date'])
print(surveys.dtypes)
surveys.head()

site_id                   str
survey_date    datetime64[us]
nitrate               float64
richness                int64
dtype: object


,site_id,survey_date,nitrate,richness
0,SITE_016,2021-01-01 00:00:00.000000,1.74,34
1,SITE_014,2021-01-06 11:56:22.914572,0.98,8
2,SITE_007,2021-01-11 23:52:45.829145,1.50,25
3,SITE_010,2021-01-17 11:49:08.743718,9.76,12
4,SITE_003,2021-01-22 23:45:31.658291,2.38,26


---
## Extracting Date Components with `.dt`

In [2]:
surveys = surveys.assign(
    year      = surveys['survey_date'].dt.year,
    month     = surveys['survey_date'].dt.month,
    quarter   = surveys['survey_date'].dt.quarter,
    day_of_year = surveys['survey_date'].dt.dayofyear,
    month_name= surveys['survey_date'].dt.strftime('%B'),
    season    = surveys['survey_date'].dt.month.map({
        12:'winter', 1:'winter', 2:'winter',
        3:'spring',  4:'spring', 5:'spring',
        6:'summer',  7:'summer', 8:'summer',
        9:'autumn', 10:'autumn', 11:'autumn'
    })
)
print(surveys[['survey_date','year','month','quarter','season']].head(8))

                 survey_date  year  month  quarter  season
0 2021-01-01 00:00:00.000000  2021      1        1  winter
1 2021-01-06 11:56:22.914572  2021      1        1  winter
2 2021-01-11 23:52:45.829145  2021      1        1  winter
3 2021-01-17 11:49:08.743718  2021      1        1  winter
4 2021-01-22 23:45:31.658291  2021      1        1  winter
5 2021-01-28 11:41:54.572864  2021      1        1  winter
6 2021-02-02 23:38:17.487437  2021      2        1  winter
7 2021-02-08 11:34:40.402010  2021      2        1  winter


---
## Datetime Arithmetic and Timedeltas

In [3]:
# Days since a reference date
ref_date = pd.Timestamp('2021-01-01')
surveys['days_since_start'] = (surveys['survey_date'] - ref_date).dt.days

# Time between consecutive surveys per site
surveys_sorted = surveys.sort_values(['site_id', 'survey_date'])
surveys_sorted['days_since_prev'] = (
    surveys_sorted
    .groupby('site_id')['survey_date']
    .diff()
    .dt.days
)

print(surveys_sorted[['site_id','survey_date','days_since_prev']].head(10))

# Adding fixed offsets
surveys['followup_due'] = surveys['survey_date'] + pd.DateOffset(months=6)
surveys['followup_due'].head(3)

      site_id                survey_date  days_since_prev
19   SITE_001 2021-04-15 10:51:15.376884              NaN
29   SITE_001 2021-06-09 10:15:04.522613             54.0
73   SITE_001 2022-02-06 07:35:52.763819            241.0
117  SITE_001 2022-10-06 04:56:41.005025            241.0
121  SITE_001 2022-10-28 04:42:12.663316             21.0
172  SITE_001 2023-08-04 13:37:41.306532            280.0
6    SITE_002 2021-02-02 23:38:17.487437              NaN
41   SITE_002 2021-08-14 09:31:39.497487            192.0
42   SITE_002 2021-08-19 21:28:02.412060              5.0
44   SITE_002 2021-08-30 21:20:48.241206             10.0


0   2021-07-01 00:00:00.000000
1   2021-07-06 11:56:22.914572
2   2021-07-11 23:52:45.829145
Name: followup_due, dtype: datetime64[us]

---
## Resampling and Rolling Windows

In [4]:
# Set datetime as index for resample()
ts = surveys.set_index('survey_date').sort_index()

# Monthly mean nitrate across all sites
monthly = (
    ts['nitrate']
    .resample('ME')          # 'ME' = month end; use 'MS' for month start
    .agg(['mean', 'count'])
    .rename(columns={'mean':'nitrate_mean', 'count':'n_samples'})
)
print('Monthly aggregation:')
print(monthly.head(6))

# 90-day rolling mean
ts['nitrate_rolling90'] = (
    ts['nitrate']
    .rolling('90D', min_periods=5)
    .mean()
)
print('\nRolling mean sample:')
print(ts[['nitrate','nitrate_rolling90']].head(8))

Monthly aggregation:
             nitrate_mean  n_samples
survey_date                         
2021-01-31       3.228333          6
2021-02-28       4.306000          5
2021-03-31       5.491667          6
2021-04-30       3.544000          5
2021-05-31       3.960000          6
2021-06-30       2.626000          5

Rolling mean sample:
                            nitrate  nitrate_rolling90
survey_date                                           
2021-01-01 00:00:00.000000     1.74                NaN
2021-01-06 11:56:22.914572     0.98                NaN
2021-01-11 23:52:45.829145     1.50                NaN
2021-01-17 11:49:08.743718     9.76                NaN
2021-01-22 23:45:31.658291     2.38           3.272000
2021-01-28 11:41:54.572864     3.01           3.228333
2021-02-02 23:38:17.487437     2.97           3.191429
2021-02-08 11:34:40.402010     5.67           3.501250


---
## Time Zones

In [5]:
# Localise naive datetime to a time zone
surveys['survey_date_utc'] = surveys['survey_date'].dt.tz_localize('UTC')

# Convert to another time zone
surveys['survey_date_eastern'] = (
    surveys['survey_date_utc'].dt.tz_convert('US/Eastern')
)

# Strip time zone for storage or comparison with naive datetimes
surveys['date_naive'] = surveys['survey_date_utc'].dt.tz_localize(None)

print(surveys[['survey_date','survey_date_utc','survey_date_eastern']].head(3))
print('\nRule: always store datetimes as UTC; convert to local time only for display.')

                 survey_date                  survey_date_utc  \
0 2021-01-01 00:00:00.000000        2021-01-01 00:00:00+00:00   
1 2021-01-06 11:56:22.914572 2021-01-06 11:56:22.914572+00:00   
2 2021-01-11 23:52:45.829145 2021-01-11 23:52:45.829145+00:00   

               survey_date_eastern  
0        2020-12-31 19:00:00-05:00  
1 2021-01-06 06:56:22.914572-05:00  
2 2021-01-11 18:52:45.829145-05:00  

Rule: always store datetimes as UTC; convert to local time only for display.


---

## Common Pitfalls

**1. Leaving datetime columns as `object` dtype**  
A column of date strings stored as `object` does not support `.dt` accessor methods, sorting by date, or arithmetic. Always convert with `pd.to_datetime()` immediately after loading. Use `errors='coerce'` to identify unparseable values rather than letting them crash the parse.

**2. Mixing time-zone-aware and time-zone-naive datetimes**  
Comparing or merging a tz-aware column with a tz-naive column raises a `TypeError`. Decide on UTC as a canonical representation, store datetimes in UTC, and convert to local time only for display. Never store naive datetimes when data comes from multiple time zones.

**3. Using `resample()` without setting a DatetimeIndex**  
`resample()` requires the DataFrame's index to be a `DatetimeIndex`. Calling it on a DataFrame with a datetime column (not the index) raises a `TypeError`. Set the index with `.set_index('date_col')` first, or use `groupby(pd.Grouper(key='date_col', freq='ME'))` if you need to preserve the original integer index.

**4. Assuming `diff()` on dates returns days automatically**  
Subtracting two `datetime64` columns produces a `Timedelta` column, not an integer. To get days, call `.dt.days` on the result. Forgetting this produces a `Timedelta` that looks like it might be a number but behaves differently in arithmetic and comparisons.

**5. Using `resample('M')` without knowing end vs. start anchoring**  
`'M'` was deprecated and produces a warning; `'ME'` is month-end anchored (last day of month), `'MS'` is month-start anchored (first day). Always use the explicit alias. The choice matters when joining resampled results back to other monthly data.

---
*python_methods_library · Samantha McGarrigle · [github.com/samantha-mcgarrigle](https://github.com/samantha-mcgarrigle)*